# 04 — Compute Library Features (SOAP, Magpie/matminer)

Computes SOAP descriptors (via DScribe) and Magpie/matminer atomic features for all training structures.

## Prerequisites / Input files
- `Fe-Mo/Atomsobjects/Fe-Mo-POSCAR-initial-rescaled-AtomsObjects.pkl`
- DScribe and matminer packages (public, included in `environment_public.yaml`)

## Outputs
- `Fe-Mo/Descriptors/soap_features__*.csv`
- `Fe-Mo/Descriptors/matminer_atomic_features.pkl`
- `Fe-Mo/Descriptors/DatasetFeatures.pkl`



#  Calculation of features from available libraries

In [ ]:
from Tools.DatasetTools.Commoms import *

In [ ]:
# Skip descriptor computation if output files already exist
_lib_output = os.path.join('Fe-Mo', 'Descriptors', 'DatasetFeatures.pkl')
_lib_descriptors_exist = os.path.exists(_lib_output)
if _lib_descriptors_exist:
    print('Library descriptors found — skipping recomputation.')


In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif', )

In [ ]:
dataset = 'Fe-Mo'  # 'Cr-Co-W' # 'Fe-Mo'#
components = dataset.split('-')
system=dataset.replace('-','')
try:
    from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer
    _HAS_BOPFOX = True
except (ImportError, Exception):
    _HAS_BOPFOX = False

import Tools.DatasetTools.GeneralFeaturizer as gf

BS = pd.read_pickle(os.path.join(dataset, 'FullyCuratedParsedBriefSummary.pkl'))
AtomsObjects = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{dataset}-POSCAR-initial-rescaled-AtomsObjects.pkl')).dropna()


In [ ]:
intersection = BS.index.intersection(AtomsObjects.index)

In [ ]:
BS = BS.loc[intersection]
AtomsObjects = AtomsObjects.loc[intersection]

In [ ]:
if not _lib_descriptors_exist:
    
    PymatgenStructures = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{dataset}-POSCAR-initial-rescaled-PymatgenStructures.pkl')).dropna()
    SublatticeTags = pd.read_pickle(os.path.join(dataset,'Atomsobjects', 'SUBLATICETAGS_POSCAR.initial.pkl'))
    SublatticeSorters = pd.read_pickle(os.path.join(dataset,'Atomsobjects', 'SORTERS_POSCAR.initial.pkl'))
    SublatticeSorters.index = SublatticeSorters.index.astype(str).str.strip()
    SublatticeTags.index = SublatticeSorters.index.astype(str).str.strip()
    
    BS.dropna(inplace=True)
    
    import numpy as np

# Prepare Dataset Features

In [ ]:
if not _lib_descriptors_exist:
    from importlib.machinery import SourceFileLoader

In [ ]:
if not _lib_descriptors_exist:
    from sklearn.preprocessing import  OneHotEncoder, LabelEncoder
    encoder = LabelEncoder()

In [ ]:
if not _lib_descriptors_exist:
    Features = Featurizer(BS)

In [ ]:
if not _lib_descriptors_exist:
    DatasetCompositionFeatures = Features.get_fractions_by_components()

In [ ]:
if not _lib_descriptors_exist:
    #DatasetFeatures = pd.concat((DatasetCompositionFeatures, DatasetMagneticFeature, StructureNameFeature), axis=1)
    pass

##  Magnetism and structure

In [ ]:
if not _lib_descriptors_exist:
    structure_name_encoder = LabelEncoder()
    StructureNameFeature = BS.Phase
    StructureNameFeature.name='Structure'
    structure_name_encoder.fit(StructureNameFeature)
    DatasetStructureFeature = pd.Series(structure_name_encoder.transform(StructureNameFeature), name='Structure', index = StructureNameFeature.index)

In [ ]:
if not _lib_descriptors_exist:
    MagneticFeature = Features.MagFeature

In [ ]:
if not _lib_descriptors_exist:
    magnetic_encoder = LabelEncoder()
    MagneticFeature.name = 'Mag'
    magnetic_encoder.fit(MagneticFeature)
    DatasetMagneticFeature = pd.Series(magnetic_encoder.transform(MagneticFeature), name='Mag', index = StructureNameFeature.index)

In [ ]:
if not _lib_descriptors_exist:
    DatasetMagneticFeature

In [ ]:
if not _lib_descriptors_exist:
    DatasetFeatures = pd.concat([DatasetMagneticFeature, DatasetStructureFeature, DatasetCompositionFeatures, BS.num_atoms], axis = 1)

In [ ]:
if not _lib_descriptors_exist:
    DatasetCompositionFeatures

## Coordination Polyhedra feature

The first feature that we would like to have is the count of each CP in each sample. for that we construct a vector in the following way:

$$ N_{CN}^i = \#^i CP $$

Next feature we want is the composition in each CP. for this we choose to represent the elment numerically by their atomic numbers, and the CP-resolved composition becomes the average atomc numbers,

$$ Z_{CP} ^i = \dfrac{1}{n_{at}^i} \sum_{at \in CP} Z_{at} $$

In [ ]:
if not _lib_descriptors_exist:
    SublatticeSorters

In [ ]:
if not _lib_descriptors_exist:
    gf = SourceFileLoader('gf','Tools/DatasetTools/GeneralFeaturizer.py').load_module()

In [ ]:
if not _lib_descriptors_exist:
    SortingFeatures = gf.sorting_feature(AtomsObjects, SublatticeSorters, SublatticeTags)

In [ ]:
if not _lib_descriptors_exist:
    SortingFeatures.sorters = gf.correct_sortings_fromphases(AtomsObjects.loc[BS.index], BS.Phase, SortingFeatures.sorters[BS.index])
    SortingFeatures.sublatticetags = gf.correct_occupation_fromphases(BS.Phase, SortingFeatures.sublatticetags, AtomsObjects.atoms)
    sampleinspecial = BS.Phase.map(lambda p: p in gf.specialphases)
    empty = SortingFeatures.sublatticetags.map(lambda sublat: '' in sublat)
    SortingFeatures.sublatticetags[empty] = ['A']
    wrong = SortingFeatures.sublatticetags.map(lambda sublat: 'A' not in sublat) 
    fixable = SortingFeatures.sublatticetags.loc[wrong].map(type) == np.ndarray #.map(np.unique)
    CNList = gf.get_sitecn(BS.Phase, AtomsObjects.atoms, SortingFeatures.sorters)

## Position Features

In [ ]:
if not _lib_descriptors_exist:
    elements = np.unique(BS.filter(regex='^atom_').values.ravel())
    ABOCC = pd.concat([BS.filter(regex='atom_'), Features.occupation], axis = 1)
    ABOCC.rename(columns={ABOCC.columns[-1]: 'index'}, inplace=True)

In [ ]:
if not _lib_descriptors_exist:
    Positions = {}
    for index, item in ABOCC.iterrows():
        if item['index'] == '':
            thisposition = {index: [item[f'atom_A']]*len(np.unique(gf.cn_dict[BS.Phase[index]]))}
        else:
            thisposition = {index: [item[f'atom_{occ}'] for occ in item['index'] ]}
        Positions.update(thisposition)
    Positions = pd.DataFrame.from_dict(Positions, orient='index')
    Positions[Positions.isnull()] = 0
    for i, element in enumerate(elements):
        Positions[Positions==element] = i
    Positions.columns = [f'Pos_{col+1}' for col in Positions.columns]
    #Positions[Positions.Pos_1.map(type) == str] = np.nan

## Averages over Coordination polyhedra

### Number of each CP in each structure

In [ ]:
if not _lib_descriptors_exist:
    CN = gf.featurize_series(CNList, CNList, normalization='NCP', return0 = False)
    newcolumns = ['N'+col for col in CN.columns]
    CN.columns = newcolumns

### Composition and volume of the CP

In [ ]:
if not _lib_descriptors_exist:
    from mendeleev import element

In [ ]:
if not _lib_descriptors_exist:
    AtomicNumbers=AtomsObjects.atoms.map(lambda a: a.numbers)
    AtomicNumbers.name = 'AtomicNumbers'
    symbols = dataset.split('-')
    volums = {symb: element(symb).atomic_volume for symb in symbols}

In [ ]:
if not _lib_descriptors_exist:
    AtomicVolumes = AtomsObjects.atoms.map(lambda a: np.array([volums[at] for at in a.get_chemical_symbols()]))

In [ ]:
if not _lib_descriptors_exist:
    CNList

In [ ]:
if not _lib_descriptors_exist:
    CPVol = gf.featurize_series(AtomicVolumes.loc[CNList.index], CNList, return0=False, normalization='NCP')

In [ ]:
if not _lib_descriptors_exist:
    newcolumns = ['V'+col for col in CPVol.columns]
    CPVol.columns =  newcolumns

In [ ]:
if not _lib_descriptors_exist:
    CPComp = gf.featurize_series(AtomicNumbers.loc[CNList.index], CNList, return0=False, normalization='NCP')
    newcolumns = ['Z'+col for col in CPComp.columns]
    CPComp.columns = newcolumns

## Compile all the descriptors

In [ ]:
if not _lib_descriptors_exist:
    DatasetFeatures = pd.concat([DatasetStructureFeature, DatasetMagneticFeature, DatasetCompositionFeatures, CN, CPVol, CPComp, BS.num_atoms, Positions], axis=1)
    datasetfeatureslocation = os.path.join(dataset, 'Descriptors','DatasetFeatures.pkl')
    CNListlocation = os.path.join(dataset, 'Descriptors', 'CNList.pkl')
    DatasetFeatures.to_pickle(datasetfeatureslocation)
    CNList.to_pickle(CNListlocation)

In [ ]:
if not _lib_descriptors_exist:
    DatasetFeatures.columns

# Matminer Features 

In [ ]:
if not _lib_descriptors_exist:
    from Tools.DatasetTools.GetPymatgenFeatures import get_chemical_formula#*

In [ ]:
if not _lib_descriptors_exist:
    descriptorslocation = os.path.join(dataset, 'Descriptors')
    mmflatomic = os.path.join(descriptorslocation, 'matminer_atomic_features.pkl')
    mmfdensity = os.path.join (descriptorslocation, 'matminer_density_features.pkl')
    mmfcomposition =  os.path.join (descriptorslocation,'matminer_composition_features.pkl')
    mmfstructure =  os.path.join (descriptorslocation,'matminer_structure_features.pkl')
    mmsoapfeatures = os.path.join(descriptorslocation, 'matminer_soap_features.pkl')
    
    
    BS['chemical_formula'] = get_chemical_formula(BS)

In [ ]:
if not _lib_descriptors_exist:
    if 'composition' not in BS.columns: 
        BS['composition'] = StrToComposition().featurize_dataframe(BS, "chemical_formula")['composition']

In [ ]:
if not _lib_descriptors_exist:
    if 'atoms_objects' not in BS.columns:
        BS['atoms_objects'] = PymatgenStructures

In [ ]:
if not _lib_descriptors_exist:
    AtomicFeaturesMagpie = load_features(mmflatomic, BS, which='atomic')
    DensitiFeatures= load_features(mmfdensity, BS, which='density')
    CompositionFeatures = load_features(mmfcomposition, BS, which='composition')
    # SOAP doesnt work from matminer
    # StructureFeatures = load_features(mmfstructure, BS, which='structure')

In [ ]:
if not _lib_descriptors_exist:
    AtomicFeaturesMagpie.columns = AtomicFeaturesMagpie.columns.str.replace('MagpieData ','')
    AtomicFeaturesMagpie.dropna(axis=1, inplace = True)
    AtomicFeaturesMagpie.describe()

In [ ]:
if not _lib_descriptors_exist:
    DensitiFeatures.dropna(axis=1, inplace=True)
    if DensitiFeatures.shape[1] > 0:
        DensitiFeatures.describe()

In [ ]:
if not _lib_descriptors_exist:
    CompositionFeatures.dropna(axis=1, inplace=True)
    CompositionFeatures.describe()

# SOAPFeatures

In [ ]:
if not _lib_descriptors_exist:
    from dscribe.descriptors import SOAP
    from mendeleev import element
    import ase
    from sklearn.feature_selection import VarianceThreshold

In [ ]:
if not _lib_descriptors_exist:
    atomsobjectlocation = os.path.join(dataset, 'Atomsobjects')
    atomsobjectfile = os.path.join(atomsobjectlocation,f'{dataset}-POSCAR-initial-rescaled-AtomsObjects.pkl')

In [ ]:
if not _lib_descriptors_exist:
    AtomsObjects = pd.read_pickle(atomsobjectfile).dropna().loc[intersection]

## Canonical and specific

In [ ]:
if not _lib_descriptors_exist:
    def reset_symbols(a: ase.atoms.Atoms, newsym : str = 'W'):
        newa = a.copy()
        natoms = newa.get_global_number_of_atoms()
        newsymbols = [newsym]*natoms
        newa.set_chemical_symbols(newsymbols)
        return newa

In [ ]:
if not _lib_descriptors_exist:
    soapcases = ['canonicalFe','canonicalW','specific']

In [ ]:
if not _lib_descriptors_exist:
    SOAPFEATURES = {}
    EXPANDED_SOAP = {}
    AVE_SOAP = {}
    variances = {}
    SEL_SOAP = {}
    FINAL_SOAP = {}
    soap_params = dict(
        r_cut = 4,
        n_max = 5,
        l_max = 4, # f
        sigma = 0.1,
        rbf = 'gto',
        periodic = True,
    #    crossover = True
    )
    params_str = '__'.join([f'{key}_{val}' for key, val in soap_params.items()])
    soap_features_file={}

In [ ]:
if not _lib_descriptors_exist:
    AtomsObjects = AtomsObjects.loc[BS.index]

In [ ]:
if not _lib_descriptors_exist:
    AtomsObjects

In [ ]:
if not _lib_descriptors_exist:
    for soapcase in soapcases:
        print(soapcase)
        soap_features_file.update({soapcase: os.path.join(descriptorslocation, f'soap_features__{soapcase}__{params_str}.csv')})
        if 'canonicalFe' in soapcase:
            species=[26]
            thisatomsobjects = AtomsObjects['atoms'].map(lambda a: reset_symbols(a, 'Fe'))
        elif 'canonicalW' in soapcase:
            species=[74]
            thisatomsobjects = AtomsObjects['atoms'].map(lambda a: reset_symbols(a, 'W'))
        elif 'specific' in soapcase:
            species = [element(s).atomic_number for s in dataset.split('-')]
            thisatomsobjects = AtomsObjects['atoms'].map(lambda a: copy.deepcopy(a))
        SOAPER = SOAP(species=species, **soap_params)
        SOAPFEATURES.update({soapcase: thisatomsobjects.map(SOAPER.create)}) #,pd.DataFrame(data= columns=['SOAP'])
        EXPANDED_SOAP.update({soapcase: gf.array_expansions(SOAPFEATURES[soapcase].to_frame(name='SOAP'), ['SOAP'])})
        AVE_SOAP.update({soapcase: gf.featurize_dataframe(EXPANDED_SOAP[soapcase], CNList)})
        #AVE_SOAP[soapcase].to_csv(soap_features_file[soapcase])
    #    variances.update({soapcase: {name: col.var() for name, col in AVE_SOAP[soapcase].iteritems()}})
    #    selector = VarianceThreshold(threshold=1e-9)
    #    SEL_SOAP.update({soapcase: selector.fit_transform(AVE_SOAP[soapcase])})
    #    FINAL_SOAP.update({soapcase: pd.DataFrame(data=SEL_SOAP[soapcase], columns = selector.get_feature_names_out(), index=AVE_SOAP[soapcase].index)})
    #    FINAL_SOAP[soapcase].to_csv(soap_features_file[soapcase])

In [ ]:
if not _lib_descriptors_exist:
    AVE_SOAP['specific'].shape

In [ ]:
if not _lib_descriptors_exist:
    AVE_SOAP['specific'].columns

In [ ]:
if not _lib_descriptors_exist:
    AVE_SOAP['specific'].shape

In [ ]:
if not _lib_descriptors_exist:
    variances = AVE_SOAP['specific'].var()
    #screenvariances = FINAL_SOAP['specific'].var()

In [ ]:
if not _lib_descriptors_exist:
    maxvar = variances.max()
    minvar = variances.min()
    bins = np.logspace(np.log10(minvar), np.log10(maxvar), num=100)

In [ ]:
if not _lib_descriptors_exist:
    figm, ax = plt.subplots()
    hist = ax.hist(variances, bins=bins)
    #hist = ax.hist(screenvariances, bins=bins)
    ax.set_xscale('log')
    

## SOAP on relaxed atoms

In [ ]:
if not _lib_descriptors_exist:
    AtomsObjectsRLX = pd.read_pickle(os.path.join(dataset, 'Atomsobjects', f'{dataset}-POSCAR.relaxed-all-noscaled-AtomsObjects.pkl')).dropna()

In [ ]:
if not _lib_descriptors_exist:
    AtomsObjectsRLX = AtomsObjectsRLX.loc[intersection]

In [ ]:
if not _lib_descriptors_exist:
    SOAPER = SOAP(species=species, **soap_params)

In [ ]:
if not _lib_descriptors_exist:
    SOAPFEATURES_RLX = AtomsObjectsRLX['atoms'].map(SOAPER.create)
    SOAPFEATURES_RLX.name = 'SOAP'

In [ ]:
if not _lib_descriptors_exist:
    SOAPFEATURES_RLX

In [ ]:
if not _lib_descriptors_exist:
    EXPANDED_SOAP_RLX = gf.array_expansions(SOAPFEATURES_RLX.to_frame(), [ 'SOAP' ])

In [ ]:
if not _lib_descriptors_exist:
    AVE_SOAP_RLX = gf.featurize_dataframe(EXPANDED_SOAP_RLX, CNList)

In [ ]:
if not _lib_descriptors_exist:
    
    pass

In [ ]:
if not _lib_descriptors_exist:
    AVE_SOAP_RLX

In [ ]:
if not _lib_descriptors_exist:
    from sklearn.metrics import r2_score

In [ ]:
if not _lib_descriptors_exist:
    plt.rc('text', usetex=True)
    plt.rc('font', family='serif', )

In [ ]:
if not _lib_descriptors_exist:
    for i in range(6):
        fw, fh = plt.rcParams['figure.figsize']
        fig,axs = plt.subplots(1,5, figsize=(fw*5/2, fh))
        for CN, ax in zip(['0','CN12', 'CN14', 'CN15', 'CN16'], axs):
            feature = f'SOAP_{i}_{CN}'
            x = AVE_SOAP_RLX[feature]
            y = AVE_SOAP['specific'][feature]
        #    reg = np.poly1d(np.polyfit(x, y, 1))
        #    ytilde = reg(x)
            r2 = r2_score(y, x)
            ax.scatter(x, y , ec='k')
            proj_0_range = [x.min(), x.max()] 
            ax.plot(proj_0_range, proj_0_range, 'k')
            ax.set_title(feature+f', $R^2$ = {r2:.2f}')
        axs[0].set_ylabel('guess structure')
        fig.supxlabel('DFT relaxed', fontsize=18)
        fig.tight_layout()

In [ ]:
if not _lib_descriptors_exist:
    X=    np.ravel(AVE_SOAP['specific'])
    Y=    np.ravel(AVE_SOAP_RLX)
    fig = plt.figure()
    ax = fig.add_subplot([0.25, 0.2, 0.65, 0.7])
    ax.scatter(X,Y, ec='k')
    #plt.yscale('log')
    #plt.xscale('log')
    therange = [X.min(), X.max()]
    r2 = r2_score(X,Y)
    plt.ylabel('SOAP DFT relaxed')
    plt.xlabel('SOAP guess structures')
    plt.plot(therange,therange, '-k')
    plt.title(f'SOAP, $R^2 = ${r2:.5f}', fontsize=18)
    plt.savefig('Fe-Mo/graphs/Figure_Fe-Mo_SOAP_rlx-vs-ini.png', dpi=300)
    

# Pyscal features 

In [ ]:
if not _lib_descriptors_exist:
    atomsobjectlocation = os.path.join(dataset, 'Atomsobjects')
    atomsobjectfile = os.path.join(atomsobjectlocation,f'{dataset}-POSCAR-initial-rescaled-AtomsObjects.pkl')

In [ ]:
if not _lib_descriptors_exist:
    from tqdm.auto import tqdm
    from Tools.DatasetTools import pyscalfeaturizers as pf
    from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.struct_db import struct_db
    AtomsObjects = pd.read_pickle(atomsobjectfile).dropna()

##  Coordination Numbers

In [ ]:
if not _lib_descriptors_exist:
    featurizers = [pf.pyscal_steinhardt, pf.pyscal_cn] #, get_steinhardt]
    pyscal_features = [feature.__name__ for feature in featurizers]
    
    pyscalsteinhardt = os.path.join(descriptorslocation, 'pyscal_steinhardt.kpl')
    
    if os.path.exists(pyscalsteinhardt):
        PyscalFeatures = pd.read_pickle(pyscalsteinhardt)
    else:
        PyscalFeatures = pf.featurize_many(AtomsObjects,  featurizers, colid='atoms')
        expanded_ste = pf.expand_features(PyscalFeatures.pyscal_steinhardt, 'pyscal_steinhardt')
        PyscalFeatures = pd.concat([expanded_ste, PyscalFeatures.pyscal_cn], axis=1)
        PyscalFeatures.to_pickle(pyscalsteinhardt)

In [ ]:
if not _lib_descriptors_exist:
    PyscalFeatures

## Stainhardt parameters 

From the Steinhardt parameters obtained by Pyscal library, we also want to average over the coordination polyhedra. This time we are also saving the total average for each parameter.

$$ q_{j, CP} ^i = \dfrac{1}{n_{at}^i}\sum _{at \in CP} q_{j, at} ^i $$

In [ ]:
if not _lib_descriptors_exist:
    thisFeatures = PyscalFeatures[['pyscal_steinhardt_0','pyscal_steinhardt_1']]

In [ ]:
if not _lib_descriptors_exist:
    intersect = thisFeatures.index.intersection(CNList.index)

In [ ]:
if not _lib_descriptors_exist:
    CNPyscal  = gf.featurize_many(thisFeatures.loc[intersect], CNList[intersect], [gf.cn_average])

In [ ]:
if not _lib_descriptors_exist:
    CNPyscal

In [ ]:
if not _lib_descriptors_exist:
    PyscalFeaturesFile = os.path.join(descriptorslocation,'CNAVPyscal.pkl')

In [ ]:
if not _lib_descriptors_exist:
    CNPyscal.to_pickle(PyscalFeaturesFile)

# Characterization of Features 

In [ ]:
if not _lib_descriptors_exist:
    import matplotlib.pyplot as plt
    import seaborn as sns

##  Correlations

In [ ]:
if not _lib_descriptors_exist:
    plt.rc('font',size=22)

In [ ]:
if not _lib_descriptors_exist:
    target_case = 'EF_nmhcp'

In [ ]:
if not _lib_descriptors_exist:
    BS[target_case]

In [ ]:
if not _lib_descriptors_exist:
    FeatureGroups = {'density features': DensitiFeatures, 'atomic features': AtomicFeaturesMagpie, 'composition features': CompositionFeatures, 'Dataset Features': DatasetFeatures}
    TargetCorrelations = {groupname: GroupFeatures.corrwith(BS[target_case]).abs().dropna().sort_values(ascending=False) for groupname, GroupFeatures in FeatureGroups.items()}

In [ ]:
if not _lib_descriptors_exist:
    len(TargetCorrelations)

In [ ]:
if not _lib_descriptors_exist:
    fig = plt.figure(figsize=(len(TargetCorrelations)*7, 10))
    border=0
    totalfeatures=18
    for i, (group, TargetCorr) in enumerate(TargetCorrelations.items()):
        nfeatures = len(TargetCorr[:5])
        ax = fig.add_axes([border/totalfeatures,0.2,(nfeatures)/totalfeatures,0.7])
        border +=nfeatures
        ax.bar( TargetCorr[:5].index,TargetCorr[:5].values) #, ax = ax, orient='vertical')
    axes = fig.get_axes()
    [tax.set_yticklabels(tax.get_yticklabels(), visible=False) for tax in axes[1:]]
    [tax.sharey(axes[0]) for tax in axes[:1]]
    #axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=90) 

# Plot correlations for most correlated

In [ ]:
if not _lib_descriptors_exist:
    for name, item in DensitiFeatures.iteritems():
        print(name)

In [ ]:
if not _lib_descriptors_exist:
    DensitiFeatures[DensitiFeatures.vpa < 6]

In [ ]:
if not _lib_descriptors_exist:
    sns.histplot(DensitiFeatures.vpa)

In [ ]:
if not _lib_descriptors_exist:
    fig, axes = plt.subplots(1, 3, sharey = True, figsize=(15,5))
    for (fname, feature), ax in zip(DensitiFeatures.iteritems(), axes):
        feature.hist(ax=ax)
    #    sns.histplot(feature, ax =ax)

## By hand outlier detection:

In [ ]:
if not _lib_descriptors_exist:
    selection = (FeatureGroups['density features']['packing fraction'] < 3) & (FeatureGroups['density features']['vpa']>8) &(FeatureGroups['density features']['density']<75)

In [ ]:
if not _lib_descriptors_exist:
    def target_correlation_scatters(thisgroup, selection=None):
        featurenames = TargetCorrelations[thisgroup].index.to_list()
        if selection is None:
            selection = FeatureGroups[thisgroup].index
        nplots =  min([4, len(featurenames)])
        fig, axes = plt.subplots(1, nplots,  figsize=(7*4, 10), sharey=True)
        intersect = selection.intersection(BS[target_case].index)
        for ax, thisfeature in zip(axes, featurenames[:nplots]):
            ax.scatter(FeatureGroups[thisgroup][thisfeature][intersect], BS[target_case][intersect])
            ax.set_xlabel(thisfeature)
        axes[0].set_ylabel('$\Delta E_f$')

In [ ]:
if not _lib_descriptors_exist:
    target_correlation_scatters('atomic features')

In [ ]:
if not _lib_descriptors_exist:
    target_correlation_scatters('density features')#, selection=selection)

In [ ]:
if not _lib_descriptors_exist:
    target_correlation_scatters('composition features')

In [ ]:
if not _lib_descriptors_exist:
    target_correlation_scatters('composition features')

In [ ]:
if not _lib_descriptors_exist:
    target_correlation_scatters('Dataset Features')

In [ ]:
if not _lib_descriptors_exist:
    TargetCorrelations.keys()

In [ ]:
if not _lib_descriptors_exist:
    
    pass